## OpenRouter validation — new model set, per-token pricing

Local + rented-GPU runs are slow and expensive for the full CUAD dataset (L4 measured ~17 hr /
~$14 per full N=1500 run, see `evaluation/infra_scaling_trials.md`). OpenRouter serves LLMs
per-token across many providers — no idle/hourly cost, no GPU rental — so a full run could drop to
well under an hour and **~$0.33**, if the swapped-in models hold quality.

This notebook validates the new model set on a small N to answer:
1. Does the pipeline work end-to-end against OpenRouter?
2. Is quality good enough with the new models?
3. What does a full run cost / how long (extrapolated from this run)?

Model set (current, after `gpt-oss-20b` regressed Adherence 45%→15% at N=20 vs the L4 baseline):
- Generator: `openai/gpt-oss-120b`, pinned to **Novita**
- Judge / HyDE: `meta-llama/llama-3.1-8b-instruct`, pinned to **Groq**
- Embedder: `nomic-embed-text-v2-moe` — **kept on local Ollama** (reuses notebook `00`'s cache).
  Keeping the embedder identical means any quality change is attributable to the *generator* swap.

Providers are pinned via `extra_body={"provider": {...}}` based on OpenRouter Activity-log analysis:
DeepInfra had a long tail of catastrophic judge-call stalls (530s, 150s) on long completions despite
~99.99% uptime; Groq was consistently ~500ms with no outliers. See
`evaluation/infra_scaling_trials.md` for the full breakdown.

**Embeddings reuse:** `COLLECTION_TAG` here is `nomic_cs500_co50`, matching notebook `00`. Run `00`
on the full dataset first; then any `N_SAMPLES` here reuses those embeddings (first-N samples only).

### 0. Prerequisites

- `uv sync` (this notebook uses `langchain-openai` — run `uv sync` first).
- Local Ollama running with the embedder pulled (`ollama pull nomic-embed-text-v2-moe:latest`), or
  the collections already built by notebook `00`.
- An OpenRouter API key in `.env` as `OPENROUTER_API_KEY=...` (from `openrouter.ai`).
  - For `SMOKE_TEST = True` (next cell): **no credits needed** — free models work on a $0 balance.
  - For the real run (`SMOKE_TEST = False`): load a few dollars of credits (an N=100 run is a few cents).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time

from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import PROMPT_V3, TimingCallbackHandler, build_rag_chain
from reliablerag.env import load_secrets
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import (
    get_hybrid_retriever, get_hyde_retriever, get_reranker, get_retriever,
    get_wide_hybrid_reranked_retriever,
)

### 1. Configuration

Embedder stays on local Ollama; the three LLMs point at OpenRouter's OpenAI-compatible endpoint.
`OPENROUTER_API_KEY` is read from `.env` (loaded by `load_secrets`).

In [3]:
load_secrets()

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY  = os.environ["OPENROUTER_API_KEY"]

# --- Free smoke-test toggle -------------------------------------------------------------------
# True  : use a small OpenRouter *free* model ($0, no credits needed) just to prove the wiring
#         works. Free tier caps at 50 requests/day and this pipeline makes ~3 calls/sample, so
#         keep N_SAMPLES <= ~15. Free models are DIFFERENT from the real ones, so this validates
#         plumbing (calls go out, answers come back, embeddings cache-hit) — NOT the real
#         models' quality or cost. A small 3B model is used: fast, and far less congested than
#         the popular 70b free model (which queues heavily and hangs).
# False : the real paid models (needs credits loaded) — the actual quality/cost/timing run.
SMOKE_TEST = False

EMBEDDING_MODEL = "nomic-embed-text-v2-moe:latest"        # local Ollama — reuses notebook 00's cache

if SMOKE_TEST:
    _FREE = "meta-llama/llama-3.2-3b-instruct:free"       # small/fast free model (131K ctx), all 3 roles
    GENERATOR_MODEL = JUDGE_MODEL = HYDE_MODEL = _FREE
else:
    GENERATOR_MODEL = "openai/gpt-oss-120b"               # OpenRouter — retry after gpt-oss-20b's Adh 45%->15% regression
    JUDGE_MODEL     = "meta-llama/llama-3.1-8b-instruct"  # OpenRouter
    HYDE_MODEL      = "meta-llama/llama-3.1-8b-instruct"  # OpenRouter

CHROMA_PERSIST_DIR = os.environ.get("CHROMA_PERSIST_DIR", "./data/chroma_db")

print(f"Mode                      : {'SMOKE TEST (free model, wiring only)' if SMOKE_TEST else 'REAL (paid models)'}")
print(f"Embedder (local Ollama)   : {EMBEDDING_MODEL}")
print(f"Generator (OpenRouter)    : {GENERATOR_MODEL}")
print(f"Judge (OpenRouter)        : {JUDGE_MODEL}")
print(f"HyDE (OpenRouter)         : {HYDE_MODEL}")
print(f"Chroma dir                : {CHROMA_PERSIST_DIR}")

Mode                      : REAL (paid models)
Embedder (local Ollama)   : nomic-embed-text-v2-moe:latest
Generator (OpenRouter)    : openai/gpt-oss-120b
Judge (OpenRouter)        : meta-llama/llama-3.1-8b-instruct
HyDE (OpenRouter)         : meta-llama/llama-3.1-8b-instruct
Chroma dir                : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [4]:
# Embedder on local Ollama; LLMs on OpenRouter via the OpenAI-compatible provider.
embeddings = create_embeddings("ollama", EMBEDDING_MODEL)

# Provider pins — from OpenRouter Activity-log analysis (see evaluation/infra_scaling_trials.md):
# DeepInfra's judge/HyDE calls (llama-3.1-8b-instruct) had a long tail of catastrophic stalls
# (530s, 150s) on long completions despite ~99.99% uptime; Groq was consistently ~500ms with no
# outliers. Generator (gpt-oss-120b) completions are short, so DeepInfra's slower tok/s didn't
# show the same tail in the earlier gpt-oss-20b run, but pinning to Novita removes that variable
# too (cheap, high uptime, no observed outliers for the gpt-oss family).
GENERATOR_PROVIDER = {"order": ["Novita"], "allow_fallbacks": True}
JUDGE_HYDE_PROVIDER = {"order": ["Groq"], "allow_fallbacks": True}

def openrouter_llm(model: str, provider: dict | None = None, **kwargs):
    # timeout: fail a stuck call after 60s instead of hanging (free endpoints can queue for minutes).
    # max_retries: retry transient errors / 429s a couple times with backoff.
    extra_body = {"provider": provider} if provider else {}
    return create_llm("openai", model, base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY,
                      timeout=60, max_retries=2, extra_body=extra_body, **kwargs)

# Tags let TimingCallbackHandler label the timing lines "hyde llm" / "generator llm" / "judge llm".
llm       = openrouter_llm(GENERATOR_MODEL, provider=GENERATOR_PROVIDER).with_config(tags=["generator"])
judge_llm = openrouter_llm(JUDGE_MODEL, provider=JUDGE_HYDE_PROVIDER, temperature=0).with_config(
    tags=["judge"], callbacks=[TimingCallbackHandler()]
)
hyde_llm  = openrouter_llm(HYDE_MODEL, provider=JUDGE_HYDE_PROVIDER).with_config(tags=["hyde"])

### 2. Load a small CUAD slice

`N_SAMPLES` follows the toggle: **5** in smoke-test mode (to stay under the free daily cap), **20**
for the real run. For the real run you can bump it to 200 or any number — it reuses notebook `00`'s
pre-embedded collections for the first N samples.

In [5]:
N_SAMPLES = 5 if SMOKE_TEST else 1500   # full-dataset run (date fix confirmed at N=20, N=100, N=500)

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))
print(f"Loaded {len(samples)} CUAD samples")

Loaded 1500 CUAD samples


### 3. Run the model set — timed

Same Experiment N config (HyDE, chunk 500/50, top-20), now on `PROMPT_V3` — PROMPT_V2 explicitly
told the generator to answer bare "absent"/"no" without describing context on negative answers,
which drove Util/Comp to 0.0 on those samples regardless of correctness (see
`evaluation/infra_scaling_trials.md`). PROMPT_V3 requires every answer, positive or negative, to
cite a quote or section reference. Embedding is local (cache hit if `00` was run); generation/HyDE
calls go to OpenRouter. Pay-as-you-go means no throttle, so per-sample timing here is clean and
extrapolates to the full run.

In [6]:
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"nomic_cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"   # matches notebook 00 → reuses pre-embedded collections

# Wide-fetch + rerank: the prior hybrid (top-20 fetch per retriever, RRF) only got the correct
# chunk into the candidate pool for 39% of date-extraction questions (diagnosed against 56 CUAD
# date questions) — RRF fusion doesn't help if neither retriever's shallow top-20 fetch surfaces
# the chunk in the first place. Widening each retriever's independent fetch to top-100, unioning,
# then cross-encoder reranking to top-20 recovered 77% (vs 30% for the old hybrid RRF retriever).
FETCH_K = 100
reranker = get_reranker()  # BAAI/bge-reranker-base

t0 = time.perf_counter()
results = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_wide_hybrid_reranked_retriever(
        vs, chunks, reranker, fetch_k=FETCH_K, top_n=TOP_K
    ),
    embeddings=embeddings,
    generator_llm=llm,
    prompt_template=PROMPT_V3,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"wide-hybrid+rerank (fetch-{FETCH_K}, top-{TOP_K})",
)
gen_elapsed = time.perf_counter() - t0
print(f"\n[BENCHMARK] Generation phase — {gen_elapsed:.1f}s total, {gen_elapsed/len(samples):.2f}s/sample")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[1/1500] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 3.403s  (cache hit)
[timing] retriever: 1.199s
[timing] retriever: 0.001s
[timing] wide-hybrid+rerank (fetch-100, top-20) : 2.396s
[timing] generator llm : 5.311s
  our: The contract does **not** contain any provision requiring a party to deposit its source code into escrow with a third‑party releasable upon events such as bankruptcy or insolvency. The excerpts provided cover termination triggers (c‑d, 11.2), confidentiality (ARTICLE X, §14.3), non‑solicitation, expenses, and other miscellaneous clauses, but none mention “source code,” “escrow,” or any similar licensing/escrow arrangement.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/1500] Does the contract contain a license granted by one

In [7]:
t0 = time.perf_counter()
agg = evaluate_results(results, judge_llm, n_runs=1)
judge_elapsed = time.perf_counter() - t0
print(f"\n[BENCHMARK] Judge phase — {judge_elapsed:.1f}s total, {judge_elapsed/len(results):.2f}s/sample")
print(f"OpenRouter set (N={len(results)}) — Rel {agg['avg_relevance']:.3f} / Util {agg['avg_utilization']:.3f} / Comp {agg['avg_completeness']:.3f} / Adh {agg['adherence_rate']:.0%}")

[timing] judge llm     : 2.062s
[1/1500] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response claims that the contract does not contain any provision requiring a party to deposit its source code into escrow with a third-party releasable upon events such as bankruptcy or insolvency. The response also states that the excerpts provided cover various clauses, but none mention "source code," "escrow," or any similar licensing/escrow arrangement. Upon reviewing the documents, we can see that the contract does not explicitly mention source code or escrow, but it does cover termination triggers, confidentiality, and miscellaneous clauses. The response is partially supported by the documents, as it correctly states that the contract does not contain any explicit provision for source code escrow. However, the response also implies that the contract does not cover any licensing or escrow arrangements, which is not entirely accurate.

AttributeError: 'NoneType' object has no attribute 'get'

In [8]:
# RESUME after the AttributeError crash at sample 844 (index 843), fixed in evaluation.py —
# a judge response that parses as valid JSON but isn't a dict (e.g. bare "null") crashed
# _compute_scores instead of being treated as a parse failure. `results[:843]` already has its
# our_* fields filled in from the partial run above (evaluate_results mutates in place); only
# `results[843:]` still needs scoring. %autoreload has already picked up the evaluation.py fix.
RESUME_FROM = 843
remaining = results[RESUME_FROM:]
print(f"Resuming judge phase from sample {RESUME_FROM + 1}/{len(results)} — {len(remaining)} remaining")

t0 = time.perf_counter()
evaluate_results(remaining, judge_llm, n_runs=1)
judge_elapsed = time.perf_counter() - t0
print(f"\n[BENCHMARK] Judge phase (resumed) — {judge_elapsed:.1f}s total, {judge_elapsed/len(remaining):.2f}s/sample")

n = len(results)
agg = {
    "adherence_rate":   sum(r["our_adherence"]    for r in results) / n,
    "avg_relevance":    sum(r["our_relevance"]    for r in results) / n,
    "avg_utilization":  sum(r["our_utilization"]  for r in results) / n,
    "avg_completeness": sum(r["our_completeness"] for r in results) / n,
}
print(f"OpenRouter set (N={n}) — Rel {agg['avg_relevance']:.3f} / Util {agg['avg_utilization']:.3f} / Comp {agg['avg_completeness']:.3f} / Adh {agg['adherence_rate']:.0%}")

Resuming judge phase from sample 844/1500 — 657 remaining
[timing] judge llm     : 1.474s
[1/657] [PASS] The date of the contract...
  Adherence   : PASS  — The response as a whole is supported by the documents. The response accurately quotes the first sentence of the contract, which specifies the date of the contract. The response does not make any claims that are not supported by the documents.
  Relevance   : 0.037 — The documents provided are a contract between HOVIONE and INTERSECT ENT, Inc. The relevant information for answering the question is contained in the first sentence of the contract, which specifies the date of the contract.
  Utilization : 0.037
  Completeness: 1.000

[timing] judge llm     : 75.164s
[2/657] [FAIL] Is a party restricted from contracting or soliciting customers or part...
  Adherence   : FAIL  — 
  Relevance   : 0.053 — The documents provided are a contract between HealthCare Capital Corp. and the Sponsor, outlining the terms and conditions of their agre

### 4. Quality spot-check

The aggregate metrics don't tell you *why* an answer passed or failed. Eyeball a few of our answers
against the references to judge whether `gpt-oss-120b` is actually a good enough generator before
paying for a full run.

In [9]:
for r in results[:10]:
    print(f"Q   : {r['question'][:100]}")
    print(f"ours: {r['our_response'][:300]}")
    print(f"ref : {r['ref_response'][:300]}")
    print("-" * 100)

Q   : Is one party required to deposit its source code into escrow with a third party, which can be releas
ours: The contract does **not** contain any provision requiring a party to deposit its source code into escrow with a third‑party releasable upon events such as bankruptcy or insolvency. The excerpts provided cover termination triggers (c‑d, 11.2), confidentiality (ARTICLE X, §14.3), non‑solicitation, exp
ref : No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.
----------------------------------------------------------------------------------------------------
Q   : Does the contract contain a license granted by one party to its counterparty?
ours: No. The excerpted provisions include Counterparts, Waiver, Further Assurances, Severability, Termination, Successors & Assigns, Governing Law, Entire Agreement, Amendments, etc., bu

### 5. Cost projection to full dataset

This is the number that decides whether to commit to a full run. We don't have exact token counts
wired into the result dicts, so this uses rough per-sample estimates — replace them with the real
usage OpenRouter reports (Activity page / generation stats) for a precise figure.

OpenRouter pricing (per M tokens, pinned providers):
- `openai/gpt-oss-120b` (Novita): $0.05 in / $0.25 out
- `meta-llama/llama-3.1-8b-instruct` (Groq): $0.05 in / $0.08 out

In [10]:
FULL_N = 1500

# Rough per-sample token estimates — refine with OpenRouter's reported usage.
GEN_IN, GEN_OUT     = 3000, 250   # generator: prompt + 20 chunks in, short answer out
JUDGE_IN, JUDGE_OUT = 4000, 400   # judge: 20 chunks + answer + rubric in, scores+explanations out
HYDE_IN, HYDE_OUT   = 150, 120    # hyde: question in, short hypothetical clause out (runs once/sample)

PRICE = {  # (input $/M, output $/M) — pinned-provider list prices
    "gen":   (0.05, 0.25),   # openai/gpt-oss-120b on Novita
    "judge": (0.05, 0.08),   # meta-llama/llama-3.1-8b-instruct on Groq
    "hyde":  (0.05, 0.08),   # meta-llama/llama-3.1-8b-instruct on Groq
}

def cost(n_in, n_out, key):
    pin, pout = PRICE[key]
    return (n_in * pin + n_out * pout) / 1_000_000

per_sample = cost(GEN_IN, GEN_OUT, "gen") + cost(JUDGE_IN, JUDGE_OUT, "judge") + cost(HYDE_IN, HYDE_OUT, "hyde")
print(f"Estimated cost / sample   : ${per_sample:.4f}")
print(f"Estimated cost / full run : ${per_sample * FULL_N:.2f}  (N={FULL_N})")
print(f"vs L4 baseline            : ~$14 / full run")

Estimated cost / sample   : $0.0005
Estimated cost / full run : $0.69  (N=1500)
vs L4 baseline            : ~$14 / full run


### Decision

- **Quality holds** (metrics + spot-check acceptable) **and cost/timing acceptable** → run the full
  dataset on OpenRouter. Pin a provider for a deterministic cost/speed number, and check whether
  prompt caching lowers the effective cost further.
- **Quality drops** → step the generator up (e.g. `openai/gpt-oss-120b`, or keep your original
  `mistralai/mistral-small-3.2-24b-instruct` which needs no re-validation) and re-check.

Record the outcome in `evaluation/infra_scaling_trials.md`.